In [ ]:
# Install gwexpy with pinned versions of core dependencies for reproducibility on Colab

%pip install -q "gwexpy[all]" "gwpy<5.0.0" "numpy<2.0.0" "scipy<1.13.0" "astropy<7.0.0"

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
# ケーススタディ: イベント同期解析

このチュートリアルでは、実務でのワークフローに即した例を紹介します。ログからのイベント抽出、時刻補正（模した例）、そしてイベントごとの解析・プロット生成までの一連の流れを `SegmentTable` で管理します。

このケーススタディでも、イベント窓は `gwpy.segments.Segment` で表し、gwexpy は `SegmentTable` による表管理と `apply()` ベースの一括処理を担当します。GWpy 基盤クラスと gwexpy 拡張の役割分担は [SegmentTable: 基本](intro_segment_table.ipynb) と同じです。


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    import pandas as pd
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. イベントの同定をシミュレート
    events_df = pd.DataFrame({
    "gps": [1234567890.1, 1234567895.5, 1234567900.2],
    "snr": [15, 8, 22]
    })

    # 2. イベントから SegmentTable を作成
    # 各 GPS 時刻の前後 2 秒間 (計4秒) をセグメントとする
    segs = [Segment(t-2, t+2) for t in events_df["gps"]]
    st = SegmentTable.from_segments(segs, snr=events_df["snr"])
    st


## バッチプロット生成

`apply()` を使って、各イベントのズームプロットなどを一括生成し、そのファイルパスをテーブル自体に保存できます。

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

def generate_event_plot(row):
    # 画一的なプロット生成プロセスのSimulation
    path = f"outputs/event_{row.index}.png"
    # 実際には row["raw"].plot() などを使用
    with open(path, "w") as f: f.write("Dummy PNG")
    return {"plot_path": path}

st_plots = st.apply(generate_event_plot)
st_plots.display()

## 結果の絞り込みと出力

最後に、特定の条件（SNR > 10 など）でテーブルをフィルタリングし、結果をエクスポートします。

In [ ]:
st_best = st_plots.select(mask=st_plots.to_pandas()["snr"] > 10)
st_best.to_pandas()